# 1. Titulo y proposito

Este notebook construye un **dataset tabular autosuficiente** para Gradix a partir de fotografias de cartas organizadas en carpetas. El objetivo es documentar, de forma academica y reproducible, un flujo simplificado pero coherente con la idea central del proyecto:

**carga -> deteccion/aislamiento -> warp -> extraccion de features -> construccion tabular -> exportacion a CSV**.

A diferencia de la app o del pipeline del repositorio, este notebook **no importa modulos internos del proyecto** y no depende de `app.py`, `generar_dataset.py` ni `src.pipeline.card_analysis`. Toda la logica necesaria queda incorporada directamente en las celdas.

# 2. Fuente de datos y alcance

El notebook asume una carpeta configurable de entrada con imagenes propias de cartas, por ejemplo una estructura como:

```text
data/raw/
  damaged/
    foto_001.jpg
    foto_002.jpg
  undamaged/
    foto_101.jpg
    foto_102.jpg
```

La etiqueta inicial se obtiene de la carpeta padre inmediata. Por tanto:

- `label_condition` proviene de la estructura de carpetas.
- `target_damaged` es un target operacional binario derivado de esa etiqueta.
- `analysis_status` y `usable_for_condition_model` se calculan a partir del procesamiento tecnico de la imagen.

El flujo implementado aqui es una **adaptacion autosuficiente y simplificada** de la logica conceptual de Gradix. No pretende ser una copia exacta del repositorio, sino una version ejecutable, clara y metodologicamente defendible con dependencias minimas.

# 3. Configuracion del entorno

Solo se usan dependencias comunes: `Python`, `numpy`, `pandas`, `cv2` y `matplotlib`. Ajusta `INPUT_DIR` y `OUTPUT_CSV` segun tu entorno.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.axisbelow'] = True

# Ajusta estas rutas segun tu entorno.
INPUT_DIR = Path('data/raw')
OUTPUT_CSV = Path('data/processed/dataset_autosuficiente_gradix.csv')
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
TARGET_ASPECT_RATIO = 0.714

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f'No existe la carpeta de entrada: {INPUT_DIR}. Ajusta INPUT_DIR antes de ejecutar el notebook.'
    )

image_paths = sorted(
    [path for path in INPUT_DIR.rglob('*') if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS]
)

print('INPUT_DIR =', INPUT_DIR)
print('OUTPUT_CSV =', OUTPUT_CSV)
print('Imagenes detectadas =', len(image_paths))

preview_df = pd.DataFrame({'relative_path': [str(path.relative_to(INPUT_DIR)) for path in image_paths[:10]]})
display(preview_df)

# 4. Vista general del pipeline tecnico

El pipeline autosuficiente sigue estas etapas:

1. **Carga** de imagen con OpenCV.
2. **Deteccion / aislamiento** de la carta mediante bordes y contornos.
3. **Rectificacion** mediante warp de perspectiva cuando se obtienen cuatro esquinas plausibles.
4. **Validacion post-warp** con reglas simples sobre aspecto y visibilidad de bordes.
5. **Extraccion de features** visuales y geometricas.
6. **Construccion de una fila tabular** por imagen.
7. **Generacion del dataset completo** y exportacion a CSV.

La implementacion intenta conservar el flujo conceptual de Gradix, pero con una complejidad apropiada para un notebook autosuficiente.

In [ ]:
def list_image_files(root: Path) -> List[Path]:
    return sorted([path for path in root.rglob('*') if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS])


def infer_label_from_path(path: Path, root: Path) -> str:
    try:
        relative = path.relative_to(root)
        if len(relative.parts) >= 2:
            return relative.parts[0].lower().strip()
    except Exception:
        pass
    return 'unknown'


def normalize_label_condition(label: str) -> str:
    label = (label or '').lower().strip()
    if label in {'damaged', 'damage', 'danada', 'dañada', 'rota', 'rotas'}:
        return 'damaged'
    if label in {'undamaged', 'clean', 'sana', 'ok', 'sin_danio', 'sin_daño'}:
        return 'undamaged'
    return 'unknown'


def target_from_label(label: str):
    if label == 'damaged':
        return 1
    if label == 'undamaged':
        return 0
    return None


def ensure_bgr_uint8(image: np.ndarray) -> np.ndarray:
    if image is None:
        raise ValueError('image is None')
    if not isinstance(image, np.ndarray):
        raise TypeError('image must be a numpy.ndarray')
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if image.ndim != 3 or image.shape[2] != 3:
        raise ValueError('image must have shape (H, W, 3)')
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    return image


def order_points(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=np.float32).reshape(4, 2)
    s = points.sum(axis=1)
    diff = np.diff(points, axis=1).reshape(-1)
    top_left = points[np.argmin(s)]
    bottom_right = points[np.argmax(s)]
    top_right = points[np.argmin(diff)]
    bottom_left = points[np.argmax(diff)]
    return np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.float32)


def polygon_area(points: np.ndarray) -> float:
    pts = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    x = pts[:, 0]
    y = pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


def draw_quad_overlay(image_bgr: np.ndarray, points: Optional[np.ndarray], color=(0, 255, 0), thickness: int = 3) -> np.ndarray:
    output = image_bgr.copy()
    if points is None:
        return output
    pts = np.asarray(points, dtype=np.int32).reshape(-1, 1, 2)
    cv2.polylines(output, [pts], True, color, thickness)
    return output


def bgr_to_rgb(image_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(ensure_bgr_uint8(image_bgr), cv2.COLOR_BGR2RGB)


def show_images(items: List[Tuple[str, np.ndarray]], cols: int = 3, figsize=(15, 9)) -> None:
    if not items:
        print('No hay imagenes para mostrar.')
        return
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (title, image) in zip(axes.ravel(), items):
        if image.ndim == 2:
            ax.imshow(image, cmap='gray')
        else:
            ax.imshow(bgr_to_rgb(image))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


raw_index = pd.DataFrame(
    {
        'relative_path': [str(path.relative_to(INPUT_DIR)) for path in image_paths],
        'folder_label_raw': [infer_label_from_path(path, INPUT_DIR) for path in image_paths],
    }
)
raw_index['label_condition'] = raw_index['folder_label_raw'].map(normalize_label_condition)
raw_index['target_damaged'] = raw_index['label_condition'].map(target_from_label)
display(raw_index.head(10))

In [ ]:
def warp_from_corners(image_bgr: np.ndarray, corners: np.ndarray, target_aspect_ratio: float = TARGET_ASPECT_RATIO, min_output_height: int = 700, max_output_height: int = 1200) -> Dict[str, object]:
    image_bgr = ensure_bgr_uint8(image_bgr)
    ordered = order_points(corners)
    top_width = float(np.linalg.norm(ordered[1] - ordered[0]))
    bottom_width = float(np.linalg.norm(ordered[2] - ordered[3]))
    left_height = float(np.linalg.norm(ordered[3] - ordered[0]))
    right_height = float(np.linalg.norm(ordered[2] - ordered[1]))

    raw_width = max(1.0, (top_width + bottom_width) / 2.0)
    raw_height = max(1.0, (left_height + right_height) / 2.0)

    out_height = int(np.clip(round(raw_height), min_output_height, max_output_height))
    out_width = max(300, int(round(out_height * target_aspect_ratio)))

    destination = np.array([
        [0, 0],
        [out_width - 1, 0],
        [out_width - 1, out_height - 1],
        [0, out_height - 1],
    ], dtype=np.float32)

    transform = cv2.getPerspectiveTransform(ordered.astype(np.float32), destination)
    warped = cv2.warpPerspective(
        image_bgr,
        transform,
        (out_width, out_height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE,
    )

    return {
        'warped_image': warped,
        'ordered_corners': ordered,
        'transform_matrix': transform,
        'metrics': {
            'raw_width': float(raw_width),
            'raw_height': float(raw_height),
            'output_width': int(out_width),
            'output_height': int(out_height),
            'target_aspect_ratio': float(target_aspect_ratio),
        },
    }


def detect_card_contour(image_bgr: np.ndarray) -> Dict[str, object]:
    image_bgr = ensure_bgr_uint8(image_bgr)
    h, w = image_bgr.shape[:2]
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 60, 180)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_score = -1.0
    best_used_fallback = True

    for contour in contours:
        contour_area = float(cv2.contourArea(contour))
        if contour_area <= 0:
            continue

        area_ratio = contour_area / float(max(1, h * w))
        if area_ratio < 0.08 or area_ratio > 0.98:
            continue

        perimeter = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.02 * perimeter, True)

        if len(approx) == 4:
            points = approx.reshape(4, 2).astype(np.float32)
            used_fallback = False
        else:
            rect = cv2.minAreaRect(contour)
            points = cv2.boxPoints(rect).astype(np.float32)
            used_fallback = True

        ordered = order_points(points)
        width_est = max(1.0, float(np.linalg.norm(ordered[1] - ordered[0]) + np.linalg.norm(ordered[2] - ordered[3])) / 2.0)
        height_est = max(1.0, float(np.linalg.norm(ordered[3] - ordered[0]) + np.linalg.norm(ordered[2] - ordered[1])) / 2.0)
        aspect_ratio = width_est / height_est
        aspect_quality = max(0.0, 1.0 - abs(aspect_ratio - TARGET_ASPECT_RATIO) / 0.18)

        xs = ordered[:, 0]
        ys = ordered[:, 1]
        bbox_area = max(1.0, float((xs.max() - xs.min()) * (ys.max() - ys.min())))
        rectangularity = min(1.0, contour_area / bbox_area)

        center = ordered.mean(axis=0)
        image_center = np.array([w / 2.0, h / 2.0], dtype=np.float32)
        center_distance = float(np.linalg.norm(center - image_center))
        max_distance = float(np.linalg.norm(image_center))
        center_score = max(0.0, 1.0 - center_distance / max(1e-6, max_distance))

        area_score = min(1.0, area_ratio / 0.55)
        score = 0.45 * area_score + 0.25 * aspect_quality + 0.20 * center_score + 0.10 * rectangularity

        if score > best_score:
            best_score = score
            best = {
                'contour': contour,
                'corners': ordered,
                'area_ratio': area_ratio,
                'aspect_ratio': aspect_ratio,
                'rectangularity': rectangularity,
                'center_score': center_score,
            }
            best_used_fallback = used_fallback

    debug_images = {
        'gray': gray,
        'edges': edges,
        'closed': closed,
    }

    if best is None:
        debug_images['detected_contour_overlay'] = image_bgr.copy()
        return {
            'success': False,
            'contour': None,
            'corners': None,
            'used_fallback': True,
            'metrics': {
                'best_score': 0.0,
                'coverage_ratio': 0.0,
                'aspect_ratio': None,
                'rectangularity': 0.0,
                'center_score': 0.0,
            },
            'debug_images': debug_images,
        }

    debug_images['detected_contour_overlay'] = draw_quad_overlay(image_bgr, best['corners'])

    return {
        'success': True,
        'contour': best['contour'],
        'corners': best['corners'],
        'used_fallback': bool(best_used_fallback),
        'metrics': {
            'best_score': float(best_score),
            'coverage_ratio': float(best['area_ratio']),
            'aspect_ratio': float(best['aspect_ratio']),
            'rectangularity': float(best['rectangularity']),
            'center_score': float(best['center_score']),
        },
        'debug_images': debug_images,
    }

In [ ]:
def validate_rectified_card(warped_bgr: np.ndarray) -> Dict[str, object]:
    warped_bgr = ensure_bgr_uint8(warped_bgr)
    gray = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]

    aspect_ratio = w / float(max(1, h))
    aspect_deviation = abs(aspect_ratio - TARGET_ASPECT_RATIO)
    aspect_score = max(0.0, 1.0 - aspect_deviation / 0.18)

    border_depth = max(4, int(min(h, w) * 0.03))
    inner_depth = max(border_depth + 1, int(min(h, w) * 0.06))

    top_outer = gray[:border_depth, :]
    top_inner = gray[border_depth:inner_depth, :]
    bottom_outer = gray[h - border_depth:h, :]
    bottom_inner = gray[h - inner_depth:h - border_depth, :]
    left_outer = gray[:, :border_depth]
    left_inner = gray[:, border_depth:inner_depth]
    right_outer = gray[:, w - border_depth:w]
    right_inner = gray[:, w - inner_depth:w - border_depth]

    border_diffs = [
        abs(float(np.mean(top_outer)) - float(np.mean(top_inner))),
        abs(float(np.mean(bottom_outer)) - float(np.mean(bottom_inner))),
        abs(float(np.mean(left_outer)) - float(np.mean(left_inner))),
        abs(float(np.mean(right_outer)) - float(np.mean(right_inner))),
    ]
    border_score = min(1.0, float(np.mean(border_diffs)) / 45.0)
    postwarp_score = 0.55 * aspect_score + 0.45 * border_score
    postwarp_valid = bool(postwarp_score >= 0.55 and aspect_score >= 0.35)
    retry_recommended = bool((not postwarp_valid) or postwarp_score < 0.65)

    return {
        'postwarp_valid': postwarp_valid,
        'postwarp_score': float(postwarp_score),
        'retry_recommended': retry_recommended,
        'rectified_aspect_ratio': float(aspect_ratio),
        'rectified_aspect_ratio_deviation': float(aspect_deviation),
        'rectified_aspect_ratio_score': float(aspect_score),
        'outer_border_score': float(border_score),
    }


def extract_visual_features(image_bgr: np.ndarray) -> Dict[str, float]:
    image_bgr = ensure_bgr_uint8(image_bgr)
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    h, w = image_bgr.shape[:2]
    return {
        'width_px': int(w),
        'height_px': int(h),
        'aspect_ratio': float(w / max(1.0, h)),
        'area_px': int(w * h),
        'blur_score': float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        'brightness_score': float(np.mean(gray)),
        'contrast_score': float(np.std(gray)),
    }


def extract_geometry_features(contour: np.ndarray, image_shape: Tuple[int, int, int], used_fallback: bool) -> Dict[str, float]:
    image_area = float(max(1, image_shape[0] * image_shape[1]))
    contour_area = float(abs(cv2.contourArea(contour))) if contour is not None else 0.0
    coverage_ratio = contour_area / image_area
    return {
        'coverage_ratio': float(coverage_ratio),
        'used_fallback': bool(used_fallback),
    }


def find_content_bbox(image_bgr: np.ndarray) -> Dict[str, int]:
    gray = cv2.cvtColor(ensure_bgr_uint8(image_bgr), cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    grad_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    col_profile = cv2.GaussianBlur(np.abs(grad_x).mean(axis=0).reshape(1, -1), (1, 31), 0).ravel()
    row_profile = cv2.GaussianBlur(np.abs(grad_y).mean(axis=1).reshape(-1, 1), (31, 1), 0).ravel()
    h, w = gray.shape[:2]

    x_left_zone = col_profile[int(w * 0.05):int(w * 0.35)]
    x_right_zone = col_profile[int(w * 0.65):int(w * 0.95)]
    y_top_zone = row_profile[int(h * 0.05):int(h * 0.25)]
    y_bottom_zone = row_profile[int(h * 0.75):int(h * 0.95)]

    x_min = int(w * 0.05) + int(np.argmax(x_left_zone)) if x_left_zone.size else int(w * 0.12)
    x_max = int(w * 0.65) + int(np.argmax(x_right_zone)) if x_right_zone.size else int(w * 0.88)
    y_min = int(h * 0.05) + int(np.argmax(y_top_zone)) if y_top_zone.size else int(h * 0.10)
    y_max = int(h * 0.75) + int(np.argmax(y_bottom_zone)) if y_bottom_zone.size else int(h * 0.90)

    if x_max <= x_min or y_max <= y_min:
        x_min, y_min, x_max, y_max = int(w * 0.12), int(h * 0.10), int(w * 0.88), int(h * 0.90)

    return {'x_min': int(x_min), 'y_min': int(y_min), 'x_max': int(x_max), 'y_max': int(y_max)}


def extract_centering_features(image_bgr: np.ndarray) -> Dict[str, object]:
    h, w = image_bgr.shape[:2]
    bbox = find_content_bbox(image_bgr)
    left_margin = float(bbox['x_min'])
    right_margin = float(w - bbox['x_max'])
    top_margin = float(bbox['y_min'])
    bottom_margin = float(h - bbox['y_max'])

    def balance(a: float, b: float) -> float:
        total = a + b
        if total <= 0:
            return 0.0
        return max(0.0, 1.0 - abs(a - b) / total)

    horizontal_balance = balance(left_margin, right_margin)
    vertical_balance = balance(top_margin, bottom_margin)

    return {
        'content_bbox': bbox,
        'left_margin': float(left_margin),
        'right_margin': float(right_margin),
        'top_margin': float(top_margin),
        'bottom_margin': float(bottom_margin),
        'horizontal_balance': float(horizontal_balance),
        'vertical_balance': float(vertical_balance),
        'overall_centering': float((horizontal_balance + vertical_balance) / 2.0),
    }


def draw_content_bbox(image_bgr: np.ndarray, bbox: Dict[str, int]) -> np.ndarray:
    output = image_bgr.copy()
    cv2.rectangle(output, (bbox['x_min'], bbox['y_min']), (bbox['x_max'], bbox['y_max']), (255, 255, 255), 3)
    return output


def compute_edge_features(image_bgr: np.ndarray) -> Dict[str, float]:
    gray = cv2.cvtColor(ensure_bgr_uint8(image_bgr), cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]
    band = max(4, int(min(h, w) * 0.03))
    inset = max(1, int(min(h, w) * 0.01))
    regions = {
        'top': gray[inset:inset + band, inset:w - inset],
        'bottom': gray[h - inset - band:h - inset, inset:w - inset],
        'left': gray[inset:h - inset, inset:inset + band],
        'right': gray[inset:h - inset, w - inset - band:w - inset],
    }

    per_side_scores = {}
    stds = []
    anomalies = []
    for side, region in regions.items():
        profile = region.mean(axis=0 if side in {'top', 'bottom'} else 1).astype(np.float32)
        smooth = cv2.GaussianBlur(profile.reshape(-1, 1), (1, 9), 0).ravel() if profile.size > 0 else profile
        residual = np.abs(profile - smooth) if profile.size > 0 else np.array([0.0])
        anomaly_ratio = float(np.mean(residual > max(4.0, float(np.std(residual)) * 1.5))) if residual.size > 0 else 1.0
        band_std = float(np.std(region)) if region.size > 0 else 0.0
        side_score = max(0.0, 100.0 * (1.0 - min(1.0, 0.65 * anomaly_ratio + 0.35 * max(0.0, (18.0 - band_std) / 18.0))))
        per_side_scores[f'{side}_edge_score'] = float(side_score)
        per_side_scores[f'{side}_anomaly_ratio'] = float(anomaly_ratio)
        per_side_scores[f'{side}_std_intensity'] = float(band_std)
        stds.append(band_std)
        anomalies.append(anomaly_ratio)

    edge_score = float(np.mean([per_side_scores[f'{side}_edge_score'] for side in ['top', 'bottom', 'left', 'right']]))
    edge_confidence = float(min(1.0, np.mean(stds) / 25.0)) if stds else 0.0
    per_side_scores['edge_score'] = edge_score
    per_side_scores['edge_confidence'] = edge_confidence
    per_side_scores['edge_anomaly_mean'] = float(np.mean(anomalies)) if anomalies else 1.0
    return per_side_scores


def draw_edge_overlay(image_bgr: np.ndarray) -> np.ndarray:
    output = image_bgr.copy()
    h, w = output.shape[:2]
    band = max(4, int(min(h, w) * 0.03))
    inset = max(1, int(min(h, w) * 0.01))
    color = (255, 255, 255)
    cv2.rectangle(output, (inset, inset), (w - inset, inset + band), color, 2)
    cv2.rectangle(output, (inset, h - inset - band), (w - inset, h - inset), color, 2)
    cv2.rectangle(output, (inset, inset), (inset + band, h - inset), color, 2)
    cv2.rectangle(output, (w - inset - band, inset), (w - inset, h - inset), color, 2)
    return output


def compute_corner_features(image_bgr: np.ndarray) -> Dict[str, float]:
    gray = cv2.cvtColor(ensure_bgr_uint8(image_bgr), cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]
    patch = max(16, int(min(h, w) * 0.12))
    inset = max(1, int(min(h, w) * 0.01))
    patches = {
        'top_left': gray[inset:inset + patch, inset:inset + patch],
        'top_right': gray[inset:inset + patch, w - inset - patch:w - inset],
        'bottom_left': gray[h - inset - patch:h - inset, inset:inset + patch],
        'bottom_right': gray[h - inset - patch:h - inset, w - inset - patch:w - inset],
    }

    features = {}
    scores = []
    stds = []
    for name, patch_img in patches.items():
        highlight_ratio = float(np.mean(patch_img > (patch_img.mean() + max(10.0, patch_img.std() * 1.2)))) if patch_img.size > 0 else 1.0
        roughness = float(cv2.Laplacian(patch_img, cv2.CV_64F).var()) if patch_img.size > 0 else 0.0
        texture_std = float(np.std(patch_img)) if patch_img.size > 0 else 0.0
        score = max(0.0, 100.0 * (1.0 - min(1.0, 0.45 * min(1.0, highlight_ratio / 0.20) + 0.35 * min(1.0, roughness / 500.0) + 0.20 * max(0.0, (8.0 - texture_std) / 8.0))))
        features[f'{name}_corner_score'] = float(score)
        features[f'{name}_highlight_ratio'] = float(highlight_ratio)
        features[f'{name}_roughness'] = float(roughness)
        features[f'{name}_std_intensity'] = float(texture_std)
        scores.append(score)
        stds.append(texture_std)

    features['corner_score_raw'] = float(np.mean(scores)) if scores else 0.0
    features['corner_confidence'] = float(min(1.0, np.mean(stds) / 25.0)) if stds else 0.0
    return features


def draw_corner_overlay(image_bgr: np.ndarray) -> np.ndarray:
    output = image_bgr.copy()
    h, w = output.shape[:2]
    patch = max(16, int(min(h, w) * 0.12))
    inset = max(1, int(min(h, w) * 0.01))
    color = (255, 255, 255)
    cv2.rectangle(output, (inset, inset), (inset + patch, inset + patch), color, 2)
    cv2.rectangle(output, (w - inset - patch, inset), (w - inset, inset + patch), color, 2)
    cv2.rectangle(output, (inset, h - inset - patch), (inset + patch, h - inset), color, 2)
    cv2.rectangle(output, (w - inset - patch, h - inset - patch), (w - inset, h - inset), color, 2)
    return output


def compute_surface_features(image_bgr: np.ndarray) -> Dict[str, float]:
    gray = cv2.cvtColor(ensure_bgr_uint8(image_bgr), cv2.COLOR_BGR2GRAY)
    inner = gray[int(gray.shape[0] * 0.12):int(gray.shape[0] * 0.88), int(gray.shape[1] * 0.12):int(gray.shape[1] * 0.88)]
    glare_ratio = float(np.mean(inner >= 245)) if inner.size > 0 else 0.0
    texture_std = float(np.std(inner)) if inner.size > 0 else 0.0
    surface_score = max(0.0, 100.0 * (1.0 - min(1.0, 0.60 * min(1.0, glare_ratio / 0.05) + 0.40 * max(0.0, (10.0 - texture_std) / 10.0))))
    return {
        'glare_ratio': float(glare_ratio),
        'surface_texture_std': float(texture_std),
        'surface_score': float(surface_score),
    }

In [ ]:
def flatten_dict(data: Dict[str, object], parent_key: str = '', sep: str = '_') -> Dict[str, object]:
    flat = {}
    for key, value in data.items():
        new_key = f'{parent_key}{sep}{key}' if parent_key else key
        if isinstance(value, dict):
            flat.update(flatten_dict(value, new_key, sep=sep))
        elif isinstance(value, (str, int, float, bool, np.integer, np.floating, np.bool_)) or value is None:
            flat[new_key] = value
    return flat


def evaluate_analysis_status(detection: Dict[str, object], postwarp: Dict[str, object], visual: Dict[str, float], geometry: Dict[str, float], edge: Dict[str, float], corner: Dict[str, float]) -> Dict[str, object]:
    invalid_reasons = []
    warning_reasons = []

    if not detection.get('success', False):
        invalid_reasons.append('invalid_detection')

    if not postwarp.get('postwarp_valid', False):
        invalid_reasons.append('invalid_postwarp')

    if float(visual.get('blur_score', 0.0)) < 70:
        invalid_reasons.append('blurry_image')
    if float(visual.get('brightness_score', 0.0)) < 75:
        invalid_reasons.append('too_dark')
    if float(visual.get('brightness_score', 0.0)) > 235:
        invalid_reasons.append('too_bright')
    if float(visual.get('contrast_score', 0.0)) < 20:
        invalid_reasons.append('low_contrast')
    if float(geometry.get('coverage_ratio', 0.0)) < 0.15:
        invalid_reasons.append('low_coverage')
    if float(edge.get('edge_confidence', 0.0)) < 0.35:
        invalid_reasons.append('low_edge_confidence')
    if float(corner.get('corner_confidence', 0.0)) < 0.35:
        invalid_reasons.append('low_corner_confidence')

    if not invalid_reasons:
        if float(geometry.get('coverage_ratio', 0.0)) < 0.28:
            warning_reasons.append('borderline_coverage')
        if 70 <= float(visual.get('blur_score', 0.0)) < 120:
            warning_reasons.append('moderate_blur')
        if 0.35 <= float(edge.get('edge_confidence', 0.0)) < 0.55:
            warning_reasons.append('moderate_edge_confidence')
        if 0.35 <= float(corner.get('corner_confidence', 0.0)) < 0.55:
            warning_reasons.append('moderate_corner_confidence')

    if invalid_reasons:
        return {
            'analysis_status': 'invalid_capture',
            'invalid_reasons': '|'.join(sorted(set(invalid_reasons))),
            'usable_for_condition_model': False,
        }

    if warning_reasons:
        return {
            'analysis_status': 'valid_with_warning',
            'invalid_reasons': '|'.join(sorted(set(warning_reasons))),
            'usable_for_condition_model': False,
        }

    return {
        'analysis_status': 'valid',
        'invalid_reasons': '',
        'usable_for_condition_model': True,
    }


def process_single_image(image_path: Path, raw_root: Path) -> Dict[str, object]:
    label_raw = infer_label_from_path(image_path, raw_root)
    label_condition = normalize_label_condition(label_raw)
    target_damaged = target_from_label(label_condition)

    base_row = {
        'image_filename': image_path.name,
        'label_condition': label_condition,
        'target_damaged': target_damaged,
        'procesado_exito': False,
        'analysis_status': 'not_evaluated',
        'invalid_reasons': '',
        'usable_for_condition_model': False,
    }

    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        base_row['analysis_status'] = 'invalid_capture'
        base_row['invalid_reasons'] = 'read_error'
        return {'row': base_row, 'debug_images': {}, 'blocks': {}}

    try:
        image_bgr = ensure_bgr_uint8(image_bgr)
        detection = detect_card_contour(image_bgr)
        debug_images = dict(detection.get('debug_images', {}))

        if not detection.get('success', False) or detection.get('corners') is None:
            row = dict(base_row)
            row.update({'analysis_status': 'invalid_capture', 'invalid_reasons': 'invalid_detection'})
            row.update(flatten_dict(detection.get('metrics', {}), parent_key='detection'))
            return {'row': row, 'debug_images': debug_images, 'blocks': {'detection': detection}}

        warp = warp_from_corners(image_bgr, detection['corners'])
        warped_bgr = warp['warped_image']
        postwarp = validate_rectified_card(warped_bgr)
        visual = extract_visual_features(warped_bgr)
        geometry = extract_geometry_features(detection['contour'], image_bgr.shape, detection.get('used_fallback', True))
        centering = extract_centering_features(warped_bgr)
        edge = compute_edge_features(warped_bgr)
        corner = compute_corner_features(warped_bgr)
        surface = compute_surface_features(warped_bgr)
        status = evaluate_analysis_status(detection, postwarp, visual, geometry, edge, corner)

        debug_images['warped_card'] = warped_bgr
        debug_images['warped_with_bbox'] = draw_content_bbox(warped_bgr, centering['content_bbox'])
        debug_images['edge_overlay'] = draw_edge_overlay(warped_bgr)
        debug_images['corner_overlay'] = draw_corner_overlay(warped_bgr)

        row = dict(base_row)
        row.update(status)
        row['procesado_exito'] = True
        row.update(flatten_dict(detection.get('metrics', {}), parent_key='detection'))
        row.update(flatten_dict(warp.get('metrics', {}), parent_key='warp'))
        row.update(flatten_dict(postwarp, parent_key='postwarp'))
        row.update(flatten_dict(visual, parent_key='visual'))
        row.update(flatten_dict(geometry, parent_key='geometry'))
        row.update(flatten_dict(centering, parent_key='centering'))
        row.update(flatten_dict(edge, parent_key='edge'))
        row.update(flatten_dict(corner, parent_key='corner'))
        row.update(flatten_dict(surface, parent_key='surface'))

        return {
            'row': row,
            'debug_images': debug_images,
            'blocks': {
                'detection': detection,
                'warp': warp,
                'postwarp_validation': postwarp,
                'features': {
                    'visual': visual,
                    'geometry': geometry,
                    'centering': centering,
                    'edge': edge,
                    'corner': corner,
                    'surface': surface,
                },
            },
        }

    except Exception as exc:
        row = dict(base_row)
        row['analysis_status'] = 'invalid_capture'
        row['invalid_reasons'] = f'exception:{exc}'
        return {'row': row, 'debug_images': {}, 'blocks': {'exception': str(exc)}}

# 5. Ejemplo unitario: de una imagen a una fila

La siguiente seccion toma una imagen de ejemplo, ejecuta el pipeline completo y muestra como pasa de fotografia cruda a salida tabular.

In [ ]:
if not image_paths:
    raise RuntimeError('No se encontraron imagenes para procesar en INPUT_DIR.')

sample_path = image_paths[0]
sample_result = process_single_image(sample_path, INPUT_DIR)

print('Imagen de ejemplo:', sample_path)
display(pd.Series(sample_result['row']).head(30))

blocks = sample_result['blocks']
if 'detection' in blocks:
    print('Bloques disponibles:', list(blocks.keys()))
    display(pd.Series(blocks['detection'].get('metrics', {}), name='detection_metrics'))

In [ ]:
sample_debug = sample_result['debug_images']
preview_items = [('original', cv2.imread(str(sample_path)))]
for key in ['detected_contour_overlay', 'warped_card', 'warped_with_bbox', 'edge_overlay', 'corner_overlay']:
    if key in sample_debug:
        preview_items.append((key, sample_debug[key]))
show_images(preview_items, cols=3, figsize=(16, 10))

sample_row_df = pd.DataFrame(sorted(sample_result['row'].items()), columns=['column', 'value'])
display(sample_row_df.head(60))

# 6. Generacion del dataset completo

La celda siguiente procesa todas las imagenes de la carpeta de entrada, construye una fila tabular por imagen y exporta el resultado final a CSV.

In [ ]:
dataset_rows = []
for idx, image_path in enumerate(image_paths, 1):
    result = process_single_image(image_path, INPUT_DIR)
    dataset_rows.append(result['row'])
    if idx % 50 == 0 or idx == len(image_paths):
        print(f'Procesadas {idx}/{len(image_paths)} imagenes')

df_dataset = pd.DataFrame(dataset_rows)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_dataset.to_csv(OUTPUT_CSV, index=False)

print('Dataset exportado en:', OUTPUT_CSV)
print('Shape:', df_dataset.shape)
display(df_dataset.head())

# 7. Resumen del dataset generado

Una vez generado el DataFrame final, conviene revisar su estructura, distribuciones basicas y una pequena ficha resumen del dataset.

In [ ]:
if 'df_dataset' not in globals():
    df_dataset = pd.read_csv(OUTPUT_CSV)

print('Shape del dataset:')
print(df_dataset.shape)

print('\nColumnas iniciales:')
print(df_dataset.columns.tolist()[:80])

for column_name in ['label_condition', 'target_damaged', 'analysis_status', 'usable_for_condition_model']:
    if column_name in df_dataset.columns:
        print(f'\nvalue_counts de {column_name}:')
        display(df_dataset[column_name].value_counts(dropna=False).rename_axis(column_name).reset_index(name='count'))

dataset_report = {
    'total_rows': int(len(df_dataset)),
    'total_columns': int(len(df_dataset.columns)),
    'label_distribution': df_dataset['label_condition'].value_counts(dropna=False).to_dict() if 'label_condition' in df_dataset.columns else {},
    'analysis_status_distribution': df_dataset['analysis_status'].value_counts(dropna=False).to_dict() if 'analysis_status' in df_dataset.columns else {},
    'usable_for_condition_model_distribution': df_dataset['usable_for_condition_model'].value_counts(dropna=False).to_dict() if 'usable_for_condition_model' in df_dataset.columns else {},
}

dataset_report

# 8. Limitaciones metodologicas

Este notebook es util para documentar la construccion del dataset, pero debe interpretarse con varias limitaciones:

- **Etiquetas obtenidas desde carpeta**: `label_condition` y `target_damaged` dependen de la organizacion manual de archivos y no equivalen a una certificacion profesional de grading.
- **Sesgo por fondo e iluminacion**: la deteccion, el warp y varias features dependen del contraste entre carta y fondo, asi como de la calidad de la captura.
- **Multiples fotos de la misma carta**: puede haber varias imagenes de un mismo ejemplar, por lo que las filas no necesariamente representan observaciones totalmente independientes.
- **Target operacional**: la variable objetivo sirve para entrenamiento supervisado preliminar, no para afirmar una valuacion experta de condicion.
- **Pipeline simplificado**: esta version es autosuficiente y academica; por tanto simplifica varias heuristicas del proyecto original para mantener claridad y ejecutabilidad con dependencias minimas.

Aun con estas limitaciones, el notebook permite explicar de forma transparente como una coleccion de imagenes puede transformarse en un dataset tabular reproducible para experimentos posteriores.